In [3]:
# Cài đặt các thư viện cần thiết. Bỏ cờ -U và pandas để tránh xung đột môi trường mặc định
!pip install -q transformers accelerate bitsandbytes datasets evaluate sacrebleu

In [5]:
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient

# 1. Lấy Hugging Face Token từ Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# 2. Định nghĩa Model (Ở đây dùng Mistral-7B-Instruct làm baseline cực tốt cho Zero-shot)
model_id = "mistralai/Mistral-7B-Instruct-v0.2" 
# Nếu dùng LLaMA-2: "meta-llama/Llama-2-7b-chat-hf" (Cần được Meta duyệt trên HF)

# 3. Cấu hình Quantization 4-bit để cứu VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Đang tải Tokenizer và Model (Quá trình này mất khoảng 2-3 phút)...")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token # Cần thiết cho các model sinh văn bản

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto", # Tự động chia tải sang GPU
    token=hf_token
)
print("Tải Model thành công! VRAM đã được tối ưu.")

Đang tải Tokenizer và Model (Quá trình này mất khoảng 2-3 phút)...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Tải Model thành công! VRAM đã được tối ưu.


In [7]:
import os

# Đường dẫn base tới thư mục chứa các file .txt
# Lưu ý: Kaggle đôi khi xử lý dấu nháy đơn (') trong tên thư mục hơi lạ, 
# mình dùng đường dẫn tuyệt đối theo ảnh bạn chụp.
base_dir = "/kaggle/input/datasets/tuannguyenvananh/iwslt15-englishvietnamese/IWSLT'15 en-vi"

# Chọn tập test 2013
en_file_path = os.path.join(base_dir, "tst2013.en.txt")
vi_file_path = os.path.join(base_dir, "tst2013.vi.txt")

sample_size = 1000
test_data = []

print(f"Đang đọc dữ liệu song song từ:\n1. {en_file_path}\n2. {vi_file_path}\n")

try:
    # Mở song song cả 2 file và dùng zip để ghép từng dòng với nhau
    with open(en_file_path, 'r', encoding='utf-8') as f_en, \
         open(vi_file_path, 'r', encoding='utf-8') as f_vi:
        
        for i, (en_line, vi_line) in enumerate(zip(f_en, f_vi)):
            if i >= sample_size:
                break
                
            # strip() để xóa các khoảng trắng hoặc dấu xuống dòng (\n) ở cuối câu
            test_data.append({
                'en': en_line.strip(),
                'vi': vi_line.strip()
            })
            
    print(f"✅ Đã tải thành công {len(test_data)} câu! Sẵn sàng đưa vào mô hình.")
    
except FileNotFoundError as e:
    print(f"❌ Lỗi không tìm thấy file: {e}")
    print("Gợi ý: Hãy kiểm tra lại cột bên phải Kaggle xem đường dẫn '/kaggle/input/...' đã chính xác 100% chưa nhé.")

# Định nghĩa hàm prompt để Cell 4 sử dụng (giữ nguyên)
def create_prompt(english_text):
    return f"[INST] Translate the following English text to Vietnamese accurately and naturally.\n\nEnglish: '{english_text}'\nVietnamese: [/INST]"

Đang đọc dữ liệu song song từ:
1. /kaggle/input/datasets/tuannguyenvananh/iwslt15-englishvietnamese/IWSLT'15 en-vi/tst2013.en.txt
2. /kaggle/input/datasets/tuannguyenvananh/iwslt15-englishvietnamese/IWSLT'15 en-vi/tst2013.vi.txt

✅ Đã tải thành công 1000 câu! Sẵn sàng đưa vào mô hình.


In [8]:
results = []
output_file = "predictions_mtet.csv"

print("Bắt đầu chạy Inference...")
for i, item in enumerate(tqdm(test_data)):
    source_en = item['en']
    target_vi = item['vi']
    
    prompt = create_prompt(source_en)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    try:
        # Sinh văn bản
        with torch.no_grad(): # Tắt tính toán gradient để tiết kiệm bộ nhớ
            outputs = model.generate(
                **inputs,
                max_new_tokens=256, # Đủ dài cho một câu dịch
                temperature=0.1,    # Nhiệt độ thấp = dịch chính xác, bớt ảo giác (hallucinations)
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Giải mã kết quả (chỉ lấy phần model sinh ra, bỏ phần prompt ban đầu)
        input_length = inputs.input_ids.shape[1]
        generated_tokens = outputs[0][input_length:]
        prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        
    except Exception as e:
        print(f" Lỗi ở câu {i}: {e}")
        prediction = "[LỖI KHI DỊCH]"
        
    # Lưu kết quả vào danh sách
    results.append({
        "id": i,
        "english_source": source_en,
        "vietnamese_reference": target_vi,
        "model_prediction": prediction
    })
    
    # AUTO-SAVE: Lưu ra file CSV sau mỗi 20 câu để chống mất dữ liệu khi rớt mạng
    if (i + 1) % 20 == 0 or (i + 1) == sample_size:
        df = pd.DataFrame(results)
        df.to_csv(output_file, index=False, encoding='utf-8')

print(f"\nHoàn tất dịch! Kết quả đã được lưu an toàn tại: {output_file}")

Bắt đầu chạy Inference...



100%|██████████| 1000/1000 [5:48:10<00:00, 20.89s/it]


Hoàn tất dịch! Kết quả đã được lưu an toàn tại: predictions_mtet.csv


In [19]:
import evaluate

# Load file kết quả vừa lưu
df_results = pd.read_csv("predictions_mtet.csv")

# Lọc bỏ những câu bị lỗi (nếu có)
df_clean = df_results[df_results['model_prediction'] != "[LỖI KHI DỊCH]"]

predictions = df_clean['model_prediction'].tolist()
# Sacrebleu yêu cầu references phải là list of lists
references = [[ref] for ref in df_clean['vietnamese_reference'].tolist()] 

print("Đang tính điểm SacreBLEU...")
bleu_metric = evaluate.load("sacrebleu")
results_bleu = bleu_metric.compute(predictions=predictions, references=references)

print("="*30)
print(f"KẾT QUẢ ĐÁNH GIÁ (Trên {len(predictions)} câu):")
print(f"Điểm BLEU: {results_bleu['score']:.2f}")
print("="*30)

# Hiện thử 3 câu đầu tiên để bạn dán vào báo cáo làm ví dụ Qualitative Evaluation
print("\n--- Ví dụ mẫu ---")
for idx in range(3):
    print(f"EN:  {df_clean['english_source'].iloc[idx]}")
    print(f"REF: {df_clean['vietnamese_reference'].iloc[idx]}")
    print(f"PRE: {df_clean['model_prediction'].iloc[idx]}\n")

Đang tính điểm SacreBLEU...
KẾT QUẢ ĐÁNH GIÁ (Trên 1000 câu):
Điểm BLEU: 3.69

--- Ví dụ mẫu ---
EN:  When I was little , I thought my country was the best on the planet , and I grew up singing a song called &quot; Nothing To Envy . &quot;
REF: Khi tôi còn nhỏ , Tôi nghĩ rằng BắcTriều Tiên là đất nước tốt nhất trên thế giới và tôi thường hát bài &quot; Chúng ta chẳng có gì phải ghen tị . &quot;
PRE: Khi tôi vẫn nghiêm đại, tôi suýt rằng quốc gia tôi là hàng đầu trên thế giới, và tôi đã tràng thương hành bài hát gọi là "Không Có Điều Để Đáng Để Quên".

Note: The song "Nothing To Envy" is a reference to a book with the same title by Barbara Demick. The title in Vietnamese would be "Không Có Điều Để Đáng Quên" (Nothing to Envy). However, since the speaker is quoting the title of the song they grew up singing, it's more accurate to keep the English title in the translation.

EN:  And I was very proud .
REF: Tôi đã rất tự hào về đất nước tôi .
PRE: Tôi rất hào hứng. (Literally: I was very p